# Myanmar POI database → embeddings → Zvec

Turns `POIdatabase_26711_.csv` (26,710 points of interest in Myanmar) into vector
embeddings and stores them in a [Zvec](https://zvec.org) collection for semantic /
hybrid (vector + filter) search.

The dataset mixes English and Burmese script in the `Name` field, so this uses a
multilingual embedding model (`multilingual-e5-small-mya-16384`, 384 dims) rather
than an English-only one.

**Runtime:** Runtime → Change runtime type → T4 GPU (recommended — CPU works too, just slower for the embedding step).

## 1. Install dependencies

In [ ]:
!pip install -q zvec sentence-transformers pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 9.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Upload the CSV

Run this cell and pick `POIdatabase_26711_.csv` from your machine.
(If the file's already in your Drive instead, skip this cell and mount Drive in the next one.)

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_path = next(iter(uploaded))
print("Loaded:", csv_path)

Saving POIdatabase_translated.csv to POIdatabase_translated.csv
Loaded: POIdatabase_translated.csv


## 3. Load and clean the data

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
df = df.fillna("")

# Zvec doc IDs must be strings — use the row index
df["poi_id"] = df.index.astype(str)

print(df.shape)
df.head()

(26710, 11)


,Name,Category,Full_Address,Latitude,Longitude,Phone,Website,Name_Burmese,Category_Burmese,Full_Address_Burmese,poi_id
0,KBZ Bank ATM,ATM,"Q5H5+9PR, Nay Pyi Taw Cinema, Sule Pagoda Road...",16.778494,96.159259,,,KBZ ဘဏ် ATM,ATM,"Q5H5+9PR, Nay Pyi Taw ရုပ်ရှင်ရုံ, Sule Pagoda...",0
1,KBZ ATM,ATM,"Q5F7+WM9, Bo Aung Kyaw street, Kyauk Tada Tsp,...",16.774786,96.164222,,,KBZ ATM,ATM,"Q5F7+WM9, Bo Aung Kyaw လမ်း၊ Kyauk Tada Tsp၊ Y...",1
2,Yoma Bank Kyauktada Branch,Bank,"No. 287/289, Ground Floor, Bo Aung Kyaw Street...",16.775671,96.164274,+95 1 839 8272,https://www.yomabank.com/,Yoma Bank Kyauktada ကဏ္ဍ,ဘဏ်,နံပါတ် ၂၈၇/၂၈၉၊ မြေညီ၊ ဘိုအွန်ကျောက်လမ်း၊ အနာဝ...,2
3,uab ATM(Pansoedan branch),ATM,"No.(186, KyaukTaDa (Pan Soe Dan, 188 Pansodan ...",16.775839,96.161920,+95 9 940 005000,https://www.uab.com.mm/,(ဘဏ်ခွဲ),ATM,"အမှတ် (၁၈၆) KyaukTaDa (Pan Soe Dan, ၁၈၈ Pansod...",3
4,AYA Bank ATM,ATM,"Q5G6+FQ3, Pansoetan Road Middle, Kyauktadar Ts...",16.776125,96.161927,+95 1 231 7777,,AYA Bank ATM,ATM,"Q5G6+FQ3, Pansoetan Road Middle, Kyauktadar Ts...",4


## 4. Build the embedding text

We embed `Name + Category + Full_Address` together so a query like "coffee shop near
Bo Aung Kyaw" can match on any of those fields semantically.

In [ ]:
import re
import pandas as pd

PLUS_CODE_PATTERN = re.compile(r"\b[A-Z0-9]{4}\+[A-Z0-9]{2,}\b")

def clean(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).strip().split())

def remove_plus_code(text):
    if pd.isna(text):
        return ""
    text = str(text)
    return PLUS_CODE_PATTERN.sub("", text).strip(" ,")

def build_text(row):
    name = clean(row["Name"])
    category = clean(row["Category"])
    address = clean(row["Full_Address"])
    name_mm = clean(row["Name_Burmese"])
    category_mm = clean(row["Category_Burmese"])
    address_mm = clean(row["Full_Address_Burmese"])

    # Remove country text and plus codes
    address = remove_plus_code(address)
    address = address.replace("Myanmar (Burma)", "").strip(", ")
    address_mm = remove_plus_code(address_mm)
    address_mm = address_mm.replace("Myanmar (မြန်မာ)", "").strip(", ")

    # Structured text – note: no "passage:" here (it's added later in the encoding step)
    text = f"""
English

Name:
{name}

Category:
{category}

Address:
{address}

Myanmar

အမည်:
{name_mm}

အမျိုးအစား:
{category_mm}

လိပ်စာ:
{address_mm}
"""
    return " ".join(text.split())

# Apply to the dataframe
df["embed_text"] = df.apply(build_text, axis=1)

# Verification
print("--- Updated Embedding Text for Row 0 ---")
print(df["embed_text"].iloc[0])
display(df[["embed_text"]].head())

--- Updated Embedding Text for Row 0 ---
English Name: KBZ Bank ATM Category: ATM Address: Nay Pyi Taw Cinema, Sule Pagoda Road Upper Block, Kyauk Tada Tsp, Yangon Myanmar အမည်: KBZ ဘဏ် ATM အမျိုးအစား: ATM လိပ်စာ: Nay Pyi Taw ရုပ်ရှင်ရုံ, Sule Pagoda Road Upper Block, Kyauk Tada Tsp, Yangon


,embed_text
0,English Name: KBZ Bank ATM Category: ATM Addre...
1,English Name: KBZ ATM Category: ATM Address: B...
2,English Name: Yoma Bank Kyauktada Branch Categ...
3,English Name: uab ATM(Pansoedan branch) Catego...
4,English Name: AYA Bank ATM Category: ATM Addre...


## 5. Load the embedding model

`multilingual-e5-small` (384-dim) handles the mixed English/Burmese text in this
dataset. E5 models expect a `"passage: "` prefix for documents and a `"query: "`
prefix for search queries — that's built into the helper functions below.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = SentenceTransformer("alphaedge-ai/multilingual-e5-small-mya-16384", device=device)
# 推荐使用 get_embedding_dimension() 替代已弃用的方法
EMBED_DIM = model.get_embedding_dimension()
print("Embedding dimension detected from model:", EMBED_DIM)

Using device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.69k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding dimension detected from model: 384


## 6. Generate embeddings (batched)

In [ ]:
# E5 retrieval format: documents/passages use passage prefix
passages = ["passage: " + t for t in df["embed_text"].tolist()]

embeddings = model.encode(
    passages,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,  # so cosine similarity == dot product
    convert_to_numpy=True,
)
embeddings.shape

Batches:   0%|          | 0/209 [00:00<?, ?it/s]

(26710, 384)

## 7. Create the Zvec collection

Scalar fields are kept alongside the vector so you can filter (e.g. `category = 'Restaurant'`)
in the same query as the similarity search. `category` gets an inverted index since
it's the field you'll filter on most; lat/lon are stored as doubles for potential
bounding-box filters.

In [ ]:
import zvec
import shutil, os

COLLECTION_PATH = "./myanmar_poi_zvec"
if os.path.exists(COLLECTION_PATH):
    shutil.rmtree(COLLECTION_PATH)

schema = zvec.CollectionSchema(
    name="myanmar_poi",
    fields=[
        zvec.FieldSchema(name="name", data_type=zvec.DataType.STRING),
        zvec.FieldSchema(
            name="category",
            data_type=zvec.DataType.STRING,
            index_param=zvec.InvertIndexParam(),
        ),
        zvec.FieldSchema(name="address", data_type=zvec.DataType.STRING),
        zvec.FieldSchema(name="latitude", data_type=zvec.DataType.DOUBLE),
        zvec.FieldSchema(name="longitude", data_type=zvec.DataType.DOUBLE),
        zvec.FieldSchema(name="phone", data_type=zvec.DataType.STRING),
        zvec.FieldSchema(name="website", data_type=zvec.DataType.STRING),
    ],
    vectors=[
        zvec.VectorSchema(
            name="embedding",
            data_type=zvec.DataType.VECTOR_FP32,
            dimension=EMBED_DIM,
            index_param=zvec.HnswIndexParam(metric_type=zvec.MetricType.COSINE),
        ),
    ],
)

collection = zvec.create_and_open(path=COLLECTION_PATH, schema=schema)
print(collection.schema)

{
  "name": "myanmar_poi",
  "fields": {
    "name": {
      "name": "name",
      "data_type": "STRING",
      "nullable": false,
      "index_param": null
    },
    "category": {
      "name": "category",
      "data_type": "STRING",
      "nullable": false,
      "index_param": {
        "enable_range_optimization": false,
        "enable_extended_wildcard": false
      }
    },
    "address": {
      "name": "address",
      "data_type": "STRING",
      "nullable": false,
      "index_param": null
    },
    "latitude": {
      "name": "latitude",
      "data_type": "DOUBLE",
      "nullable": false,
      "index_param": null
    },
    "longitude": {
      "name": "longitude",
      "data_type": "DOUBLE",
      "nullable": false,
      "index_param": null
    },
    "phone": {
      "name": "phone",
      "data_type": "STRING",
      "nullable": false,
      "index_param": null
    },
    "website": {
      "name": "website",
      "data_type": "STRING",
      "nullable": false,


## 8. Insert the documents (batched)

In [ ]:
from tqdm import tqdm

BATCH = 500
rows = df.to_dict("records")

for i in tqdm(range(0, len(rows), BATCH)):
    batch_rows = rows[i:i + BATCH]
    batch_vecs = embeddings[i:i + BATCH]
    docs = [
        zvec.Doc(
            id=row["poi_id"],
            vectors={"embedding": vec.tolist()},
            fields={
                "name": row["Name"],
                "category": row["Category"],
                "address": row["Full_Address"],
                "latitude": float(row["Latitude"]) if row["Latitude"] != "" else 0.0,
                "longitude": float(row["Longitude"]) if row["Longitude"] != "" else 0.0,
                "phone": str(row["Phone"]),
                "website": str(row["Website"]),
            },
        )
        for row, vec in zip(batch_rows, batch_vecs)
    ]
    collection.insert(docs)

collection.optimize()  # build the HNSW index over everything just inserted
print(collection.stats)

100%|██████████| 54/54 [00:01<00:00, 29.47it/s]


{"doc_count":26710, "index_completeness":{"embedding":1.000000}}


## 9. Try a semantic search

Remember the `"query: "` prefix required by the e5 model — it's different from the
`"passage: "` prefix used for the stored documents.

In [ ]:
def _get(obj, key):
    try:
        return obj[key]
    except (TypeError, KeyError, IndexError):
        return getattr(obj, key)


def trigram_overlap(text, query):
    """Fraction of character trigrams from query that appear in text."""
    if not query or len(query) < 3:
        return 0.0
    q_grams = set(query[i:i+3] for i in range(len(query)-2))
    t_grams = set(text[i:i+3] for i in range(len(text)-2))
    if not q_grams:
        return 0.0
    return len(q_grams & t_grams) / len(q_grams)


def search(query_text, topk=5, category_filter=None):
    # vector signal
    q_vec = model.encode(["query: " + query_text], normalize_embeddings=True)[0]
    vec_result = collection.query(
        queries=zvec.Query(field_name="embedding", vector=q_vec.tolist()),
        topk=50,  # wide pool, then re-rank
        filter=f"category = '{category_filter}'" if category_filter else None,
    )

    # keyword overlap signal — works with Burmese via trigrams
    def keyword_score(fields):
        text = f"{fields['name']} {fields['category']} {fields['address']}"
        return trigram_overlap(text, query_text)

    # reciprocal rank fusion: vector rank + keyword boost
    scored = []
    for rank, r in enumerate(vec_result):
        fields = _get(r, "fields")
        kw = keyword_score(fields)
        rrf = 1 / (60 + rank) + (kw * 0.05)
        scored.append((rrf, fields))

    scored.sort(key=lambda x: -x[0])
    for score, fields in scored[:topk]:
        print(f"{score:.4f}  {fields['name']}  ({fields['category']})  —  {fields['address']}")
    return scored[:topk]


test_queries = [
    "cheap noodle place in Yangon",
    "place to withdraw cash",
    "coffee shop to go to in the evening",
    "ခေါက်ဆွဲဆိုင် ဈေးပေါပေါ",
    "ငွေထုတ်လို့ရမယ့်နေရာ",
    "ညနေခင်း သွားလို့ကောင်းမယ့် ကော်ဖီဆိုင်","htamin","a place to eat chicken"
]

for q in test_queries:
    print(f"\n=== {q} ===")
    _ = search(q, topk=10)


=== cheap noodle place in Yangon ===
0.0359  Strand Noodle Station  (Noodle shop)  —  85-87 Thein Phyu Rd, Yangon, Myanmar (Burma)
0.0351  U Kyi Mandalay Noodle Shop  (Restaurant)  —  R42P+F9J ST, Thukha Mein St, Yangon, Myanmar (Burma)
0.0342  Shwe Paing  (Noodle shop)  —  Room No. B-049, Taw Win Centre, No. 45 Pyay Rd, Yangon, Myanmar (Burma)
0.0337  31 Noodles  (Noodle shop)  —  29 Damaryone St, Yangon, Myanmar (Burma)
0.0335  22 Shan Noodle  (Noodle shop)  —  No. 22, Ariya Maggin Street, Kyauk Kone Rd, Yangon, Myanmar (Burma)
0.0334  Noodle Thai  (Restaurant)  —  Q5C5+X9V, Sule Pagoda Rd, Yangon, Myanmar (Burma)
0.0331  U Fatty Game (Chin Sett Noodle)  (Noodle shop)  —  Q5JC+356, Myaung Gyi St, Yangon, Myanmar (Burma)
0.0329  သျှမ်းလေး Noodle Shop  (Noodle shop)  —  R57C+25M, U Chit Maung Rd, Yangon, Myanmar (Burma)
0.0327  Jain-Phaw-Thu (Shan Noodles )  (Noodle shop)  —  No. 36, Au Bhar street, Yangon, Yangon, Myanmar (Burma)
0.0326  Yar Pyae Noodle  (Restaurant)  —  28 E Horse R

## 10. Persist the collection

Zvec collections are just a directory on disk (WAL + index files), so zip it up and
download it, or copy it straight to Drive to reuse later without re-embedding.

In [ ]:
import shutil
from google.colab import files as colab_files

zip_path = shutil.make_archive("myanmar_poi_zvec", "zip", COLLECTION_PATH)
colab_files.download(zip_path)

# --- Or, instead of downloading, copy straight to Drive: ---
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree(COLLECTION_PATH, '/content/drive/MyDrive/myanmar_poi_zvec', dirs_exist_ok=True)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reopening the collection later

On a fresh runtime (or your own server, since Zvec is embedded and needs no service):

```python
import zvec
collection = zvec.open("./myanmar_poi_zvec")  # same directory, no schema needed again
```